In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import Point, Polygon
from tqdm import tqdm
import requests

In [ ]:
#Read Shape File --> The shape file gives a MultiPolygon Geometry Column
gdf = gpd.read_file(r"D:\Work\WB\LDT\countries\SRB\shapefiles\gadm41_SRB_2.json")

#Adjust for GeoSpatial Data
center = gpd.GeoDataFrame(gdf[['GID_2', 'NAME_2']])

#Change the MultiPolygon Geometry Column to make it more useful
center['geometry'] = gdf.centroid
center = center.to_crs(gdf.crs)
center['lat'] = center.geometry.y
center['lon'] = center.geometry.x

gdf = gdf[['GID_2', 'NAME_2', 'ENGTYPE_2', 'geometry']]

C:\Users\sonle\AppData\Local\Temp\ipykernel_43620\910939600.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center['geometry'] = gdf.centroid
C:\Users\sonle\AppData\Local\Temp\ipykernel_43620\910939600.py:8: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  center['geometry'] = gdf.centroid


In [43]:
pop_df = pd.read_csv(r"D:\Work\WB\LDT\countries\SRB\raw_data\Population_Serbia_2019.csv")

# Assuming your CSV has 'latitude' and 'longitude' columns, create a geometry column
pop_df['geometry'] = pop_df.apply(lambda row: Point(row['Lon'], row['Lat']), axis=1)

# Convert the DataFrame to a GeoDataFrame, specifying the coordinate reference system (CRS)
pop_gdf = gpd.GeoDataFrame(pop_df, geometry='geometry')

# Optionally, set the CRS (e.g., WGS84 which is commonly used for latitude and longitude)
pop_gdf = pop_gdf.set_crs(epsg=4326)

pop_gdf['ID'] = pop_df.index

pop_gdf = pop_gdf.rename(columns={'Lat': 'lat', 'Lon': 'lon'})

In [44]:
population_aoi = gpd.sjoin(pop_gdf, gdf, predicate='within',
                           how = 'inner')

In [45]:
population_aoi

,lat,lon,Population,geometry,ID,index_right,GID_2,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,NAME_2,VARNAME_2,NL_NAME_2,TYPE_2,ENGTYPE_2,CC_2,HASC_2
0,44.215417,19.839028,0.496571,POINT (19.83903 44.21542),0,60,SRB.7.6_1,SRB,Serbia,SRB.7_1,Kolubarski,Колубарски,Valjevo,NA,Ваљево,Opštine,Town|Municipal,NA,NA
1,44.917639,20.304028,6.588225,POINT (20.30403 44.91764),1,19,SRB.3.8_1,SRB,Serbia,SRB.3_1,GradBeograd,Београд,Palilula,NA,Палилула,Opštine,Town|Municipal,NA,NA
2,44.900972,20.282361,6.588225,POINT (20.28236 44.90097),2,27,SRB.3.16_1,SRB,Serbia,SRB.3_1,GradBeograd,Београд,Zemun,Zimony|Semlin,Земун,Opštine,Town|Municipal,NA,NA
3,44.903472,20.287083,6.588225,POINT (20.28708 44.90347),3,27,SRB.3.16_1,SRB,Serbia,SRB.3_1,GradBeograd,Београд,Zemun,Zimony|Semlin,Земун,Opštine,Town|Municipal,NA,NA
4,44.857917,20.335417,6.588225,POINT (20.33542 44.85792),4,27,SRB.3.16_1,SRB,Serbia,SRB.3_1,GradBeograd,Београд,Zemun,Zimony|Semlin,Земун,Opštine,Town|Municipal,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3666472,44.612083,21.199028,2.827553,POINT (21.19903 44.61208),3666472,8,SRB.2.5_1,SRB,Serbia,SRB.2_1,Braničevski,Браничевски,Požarevac,Pojarevaţ|Pasarofça|Passarowitz,Пожаревац,Opštine,Town|Municipal,NA,NA
3666473,44.609583,21.194306,2.827553,POINT (21.19431 44.60958),3666473,8,SRB.2.5_1,SRB,Serbia,SRB.2_1,Braničevski,Браничевски,Požarevac,Pojarevaţ|Pasarofça|Passarowitz,Пожаревац,Opštine,Town|Municipal,NA,NA
3666474,44.623472,21.195972,2.827553,POINT (21.19597 44.62347),3666474,8,SRB.2.5_1,SRB,Serbia,SRB.2_1,Braničevski,Браничевски,Požarevac,Pojarevaţ|Pasarofça|Passarowitz,Пожаревац,Opštine,Town|Municipal,NA,NA
3666475,44.619861,21.201250,2.827553,POINT (21.20125 44.61986),3666475,8,SRB.2.5_1,SRB,Serbia,SRB.2_1,Braničevski,Браничевски,Požarevac,Pojarevaţ|Pasarofça|Passarowitz,Пожаревац,Opštine,Town|Municipal,NA,NA


In [53]:
current_year = 2025
year = 2022

In [59]:
list_years = list(range(year+1, current_year+1))
year_str_contr = "|".join(map(str, list_years))
year_str_contr

'2023|2024|2025'

In [60]:
f"""asdfa"{year_str_contr}"asdfasdf"""

'asdfa"2023|2024|2025"asdfasdf'

# Healthcare Facilities

In [61]:


# Overpass API endpoint
overpass_url = "http://overpass-api.de/api/interpreter"

for year in tqdm(range(2021, 2025, 1)):
    # Overpass Query: Fetching all healthcare-related facilities in Serbia modified in 2024

    list_years = list(range(year+1, current_year+1))
    year_str_contr = "|".join(map(str, list_years))

    overpass_query = f"""
      [out:json][timeout:60];
      area["ISO3166-1"="RS"]->.searchArea;
      (
        node["healthcare"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["healthcare"!~"closed"];
        way["healthcare"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["healthcare"!~"closed"];
        relation["healthcare"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["healthcare"!~"closed"];

        node["amenity"="hospital"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        way["amenity"="hospital"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        relation["amenity"="hospital"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];

        node["amenity"="clinic"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        way["amenity"="clinic"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        relation["amenity"="clinic"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];

        node["amenity"="pharmacy"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        way["amenity"="pharmacy"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        relation["amenity"="pharmacy"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];

        node["amenity"="dentist"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        way["amenity"="dentist"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
        relation["amenity"="dentist"](area.searchArea)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
      );
      out center;
      """

    # Make the request
    response = requests.get(overpass_url, params={'data': overpass_query})
    data = response.json()
    # Convert the response to a Pandas DataFrame
    df_facilities = pd.DataFrame(data['elements'])

    # Extract the 'name', 'amenity/healthcare', 'lat', 'lon', 'type' of facility
    df_facilities['name'] = df_facilities['tags'].apply(lambda x: x.get('name', None))
    df_facilities['amenity_or_healthcare'] = df_facilities['tags'].apply(
        lambda x: x.get('amenity', x.get('healthcare', None))
    )

    # Filter necessary columns: 'id', 'lat', 'lon', 'name', 'amenity_or_healthcare', and deduplicate
    df_facilities = df_facilities[['id', 'lat', 'lon', 'name', 'amenity_or_healthcare']].drop_duplicates()
    df_facilities['year'] = year 

    print(year, len(df_facilities))

    df_facilities.to_csv(rf"D:\Work\WB\LDT\countries\SRB\raw_data\healthcare/healthcare_facilities_{year}.csv", index=False)

 25%|██▌       | 1/4 [00:34<01:43, 34.36s/it]

2021 3789


 50%|█████     | 2/4 [01:04<01:03, 31.86s/it]

2022 3789


 75%|███████▌  | 3/4 [01:36<00:31, 31.97s/it]

2023 3789


100%|██████████| 4/4 [02:13<00:00, 33.34s/it]

2024 3789


In [62]:
year_str_contr

'2025'

In [10]:
#GeoSpatial DataFrame
df_facilities = gpd.GeoDataFrame(df_facilities, geometry=gpd.points_from_xy(df_facilities.lon, df_facilities.lat))
df_facilities = df_facilities.set_crs(gdf.crs)

#Join
serbia_facilities = gpd.sjoin(df_facilities, gdf, predicate='within')

In [11]:
diversity_healthcare = list()
year = '2024'

municipalities = gdf['GID_2'].unique()

for municipality in municipalities:
  serbia_facilities_temp = serbia_facilities[serbia_facilities['GID_2'] == municipality]

  type_counts = serbia_facilities_temp['amenity_or_healthcare'].value_counts()

  # Step 2: Convert counts to proportions
  total_facilities = type_counts.sum()
  proportions = type_counts / total_facilities

  # Step 3: Calculate Shannon Diversity Index
  shannon_diversity = -np.sum(proportions * np.log(proportions))

  diversity_healthcare.append([municipality, year, shannon_diversity])

# Save into Data Frame
columns = ['GID_2', 'Year', 'healthcare-facilities-diversity']

# Create DataFrame
df = pd.DataFrame(diversity_healthcare, columns=columns)

df = df.merge(center[['GID_2', 'NAME_2']],
              how = 'left')

# Define new column order
new_column_order = ['GID_2', 'NAME_2', 'Year', 'healthcare-facilities-diversity']

# Reorder columns
df = df[new_column_order]

In [51]:
serbia_facilities['amenity_or_healthcare'].value_counts()

amenity_or_healthcare
pharmacy                 723
clinic                   129
doctors                  121
dentist                  107
laboratory                49
hospital                  47
yes                       28
physiotherapist            7
doctor                     4
medical_laboratory         3
optometrist                2
speech_therapist           2
centre                     1
podiatrist                 1
alternative                1
counselling                1
psychotherapist            1
medical_imaging            1
veterinary_pharmacy        1
nutrition_counselling      1
Name: count, dtype: int64

In [49]:
selected_hosp = serbia_facilities[serbia_facilities['amenity_or_healthcare'].isin(['hospital', 'clinic'])].rename({'amenity_or_healthcare': 'amenity'}, axis=1)

In [50]:
len(selected_hosp)

176

# Schools

In [63]:
# Define the Overpass API URL
overpass_url = "http://overpass-api.de/api/interpreter"

for year in tqdm(range(2021, 2025, 1)[:]):

    list_years = list(range(year+1, current_year+1))
    year_str_contr = "|".join(map(str, list_years))
    
    # Query to extract all schools in Serbia
    overpass_query = f"""
    [out:json];
    area["ISO3166-1"="RS"];
    (node["amenity"="school"](area)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
    way["amenity"="school"](area)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
    rel["amenity"="school"](area)["start_date"!~"{year_str_contr}"]["disused:amenity"!~"."]["abandoned:amenity"!~"."]["amenity"!~"closed"];
    );
    out center;
    """

    # Send request to Overpass API
    response = requests.get(overpass_url, params={'data': overpass_query})
    data = response.json()

    # Create DataFrame from the Overpass API response
    df_schools = pd.DataFrame(data['elements'])

    # Extract the name of the school if available
    df_schools['name'] = df_schools['tags'].apply(lambda x: x['name'] if 'name' in x.keys() else None)
    

    # # Create a GeoDataFrame with geometry points from lat/lon
    df_schools = df_schools[['id', 'lat', 'lon', 'name']].drop_duplicates()
    df_schools['year'] = year 
    print(year, year_str_contr, len(df_schools))

    df_schools.to_csv(rf"D:\Work\WB\LDT\countries\SRB\raw_data\schools/schools_{year}.csv", index=False)

 25%|██▌       | 1/4 [00:04<00:13,  4.46s/it]

2021 2022|2023|2024|2025 506


 50%|█████     | 2/4 [00:09<00:09,  4.57s/it]

2022 2023|2024|2025 506


 75%|███████▌  | 3/4 [00:13<00:04,  4.45s/it]

2023 2024|2025 506


100%|██████████| 4/4 [00:17<00:00,  4.47s/it]

2024 2025 506
